### 미세 튜닝된 LLM 평가하기

In [27]:
import psutil
import json
import urllib.request
from tqdm import tqdm

In [28]:
def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama가 실행 중이 아닙니다.")
print(f"Ollama 실행: {check_if_running('ollama')}")

Ollama 실행: True


In [29]:
# 코드 7-10 로컬 Ollama 모델에 쿼리하기
def query_model(prompt, model="Gemma4:12b", url="http://localhost:11434/api/chat"):
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "think": False,
        "stream": False,
        "options": {"seed": 123, "temperature": 0, "num_ctx": 2048}
    }

    payload = json.dumps(data).encode("utf-8")
    request = urllib.request.Request(
        # url: str,
        url,
        # data: _DataType = None,
        data=payload,
        # headers: MutableMapping[str, str] = {},
        # origin_req_host: str | None = None,
        # unverifiable: bool = False,
        # method: str | None = Non
        method="POST"
    )

    request.add_header("Content-Type", "application/json")

    response_data = ""
    with urllib.request.urlopen(request) as response:
        # while True:
            # line = response.readline().decode("utf-8")
            # if not line:
                # break
            # response_json = json.loads(line)
            # response_data += response_json["message"]["content"]
        response_json = json.loads(response.read().decode("utf-8"))
    return response_json["message"]["content"]

In [30]:
# 앞서 구현한 query_model 함수를 사용
result = query_model("What do Llamas eat?")
print(result)

Llamas are **herbivores**, meaning their diet consists entirely of plant-based foods. They are "grazers" and "browsers," meaning they eat both grass and the leaves/twigs of shrubs.

Here is a breakdown of what makes up a llama's diet:

### 1. Grasses (The Staple)
Grass is the primary component of a llama's diet. They are well-adapted to grazing in open fields and pastures. They can digest various types of grasses, including:
*   Wild grasses
*   Cultivated pasture grasses
*   Hay (dried grass)

### 2. Forage and Browse
In addition to grass, llamas enjoy "browsing," which involves eating:
*   Leaves and twigs from bushes and shrubs.
*   Herbs and wildflowers.
*   Grasses and weeds found in brushy areas.

### 3. Hay (Supplemental Feed)
In domestic settings (like farms or sanctuaries), llamas are often fed hay. This is especially important during the winter months or in areas where fresh grass is not growing. Common types of hay include:
*   Timothy hay
*   Orchard grass
*   Alfalfa (usua

In [31]:
# 테스트 세트에 있는 처음 3개의 샘플에 적용
with open("instruction-data-with-response.json", "r") as file:
    test_data = json.load(file)

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text

for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct outpt `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )
    print("\n데이터셋 응답:")
    print(">>", entry['output'])
    print("\n모델 응답:")
    print(">>", entry["model_response"])
    print("\n점수:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


데이터셋 응답:
>> The car is as fast as lightning.

모델 응답:
>> The car is as fast as a cheetah.

점수:
>> 95

-------------------------

데이터셋 응답:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

모델 응답:
>> The type of cloud associated with thunderstorms is a cumulus cloud.

점수:
>> **Score: 40**

**Explanation:**
The model's response is factually incorrect. While cumulonimbus clouds are the specific type associated with thunderstorms, "cumulus" clouds are a general category of fluffy, white clouds that do not typically produce thunderstorms unless they develop vertically into cumulonimbus clouds. Because the model provided a technically incorrect answer to a specific meteorological fact, it receives a low score.

-------------------------

데이터셋 응답:
>> Jane Austen.

모델 응답:
>> The author of 'Pride and Prejudice' is Jane Austen.

점수:
>> 95

-------------------------


In [32]:
# 코드 7-11 지시 미세 튜닝된 LLM 평가하기
def generate_model_scores(json_data):
    scores = []
    for entry in tqdm(json_data, desc="평가 항목"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry['model_response']}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt)
        # print(f"scores: {score}")
        try:
            scores.append(int(score))
        except ValueError:
            print(f"점수로 변환할 수 없습니다: {score}")
            continue

    return scores

In [33]:
# 전체 test_data에 generate_model_scores 함수를 적용해 보자.
scores = generate_model_scores(test_data)
print(f"평가 횟수: {len(test_data)}개 중 {len(scores)}개")
print(f"평균 점수: {sum(scores)/len(scores):.2f}\n")

평가 항목: 100%|██████████| 110/110 [04:10<00:00,  2.28s/it]

평가 횟수: 110개 중 110개
평균 점수: 35.82

